# QniNotebook：H₄ QSCI入力回路の構築確認

QSCI（Quantum-Selected Configuration Interaction）は、VQEなどで用意した試行状態を計算基底で測定し、頻繁に観測されるビット列を重要な電子配置として選ぶ手法です。選んだ配置が張る部分空間のハミルトニアンを古典計算機で対角化し、基底状態の近似固有値と固有ベクトルを求めます。

このデモでQniNotebookを使うのは、QSCI計算に進む前に、試行状態を作る量子回路が設計どおり組み立てられているかを確認するためです。

対象は直線に並ぶ水素H₄です。STO-3GとJordan–Wigner写像を使い、8つのスピン軌道を8量子ビットで表します。ビットの `1` は対応する軌道が占有されていることを示します。H₄は4電子なので、事後選別では電子数4、$S_z=0$ の条件を満たす状態を対象にします。

論文の実験では、Hartree–Fock状態から深さ8の $R_y$ 変分回路を作り、BFGSで最適化した状態を10,000 shots測定します。本デモでQniNotebookを使うのは、**QURI Partsで変分回路を構築した直後、VQEの最適化とサンプリングへ進む前**です。

まず全回転角を0にした既知の条件で、初期配置、ゲートの接続、途中の状態、最終状態を確認します。これはVQEやQSCIの計算結果を再現するものではなく、論文Appendix C3 / Fig. 10に示された8量子ビット変分回路と、QSCIの入力状態を測定する直前の確認工程を扱うNotebookです。

論文：https://arxiv.org/pdf/2302.11320

## このデモの目的

量子回路は、コード生成や静的解析だけでは意図どおりに動いているかを十分に判断できないことがあります。回路が長くなるほど、初期状態、ゲートの接続、パラメーター、測定のどこで問題が生じたのかを切り分けることも難しくなります。

特に状態準備回路では、回路のどの位置で状態が変わったのかを調べ、その結果をほかの人と共有できることが重要です。回路構造と、その位置まで実行した状態を対応付けて確認できれば、問題箇所をより具体的に議論できます。

そこで、**QURI Partsで構築した量子回路をNotebook上で可視化し、選択した回路位置までの状態を同じ画面で確認できれば、状態準備の意図確認と問題箇所の切り分けをしやすくなる**、という仮説を置きます。

QniNotebookは、この仮説を検証するためのプロトタイプです。

このデモでは、QSCIの入力状態を準備する回路を例に、次のことを確認します。

- 論文が初期状態として指定するHartree–Fock配置 `00001111` が作られているか
- 8回の繰返しが、それぞれ7本の隣接CNOTゲートと8本の $R_y$ ゲートで構成されているか
- 全回転角を0にした既知の条件で、測定直前の状態が `00001111` に戻るか

ここで確認できるのは、回路と状態を対応付けて表示できることです。レビュー時間や理解度が実際に改善するか、QniNotebookを使わない場合より有用かは、このデモだけでは判断できず、別途利用者評価が必要です。

## 1. 論文と同じ8量子ビット変分回路を作る

論文Section IVでは、直線H₄のVQEに深さ8の $R_y$ 変分回路を使っています。Appendix C3 / Fig. 10に示された構造は、Hartree–Fock状態を初期状態とし、最初に8本の $R_y$ ゲートを置いた後、7本の隣接CNOTゲートと8本の $R_y$ ゲートを8回繰り返すものです。すべての回転ゲートが独立したパラメーターを持つため、パラメーター数は $8\times(8+1)=72$ です。

※このデモでは、表示結果をあらかじめ予測できるよう、72個の回転角をすべて0にします。これはVQEで得られたパラメーターではなく、回路構築を確認するためのチェック用条件です。

論文Appendix C3に合わせ、量子ビット `q0` から `q3` をXゲートで占有させます。表示上のビット列は `00001111` で、左から `q7 ... q0` の順に並びます。QURI/OpenFermionの交互スピン軌道順序では4電子かつ $S_z=0$ のHartree–Fock配置に対応します。

In [2]:
from qni_jupyter import qni
from quri_parts.circuit import QuantumCircuit

QUBIT_COUNT = 8
DEPTH = 8
ZERO_PARAMETERS = [0.0] * (QUBIT_COUNT * (DEPTH + 1))


def build_h4_ry_ansatz(parameters, *, measure=False):
    if len(parameters) != QUBIT_COUNT * (DEPTH + 1):
        raise ValueError("72 parameters are required")

    circuit = QuantumCircuit(
        QUBIT_COUNT, cbit_count=QUBIT_COUNT if measure else 0
    )
    for q in range(4):
        circuit.add_X_gate(q)  # Hartree–Fock |00001111>

    parameter_index = 0
    for q in range(QUBIT_COUNT):
        circuit.add_RY_gate(q, parameters[parameter_index])
        parameter_index += 1

    for _ in range(DEPTH):
        for q in range(QUBIT_COUNT - 1):
            circuit.add_CNOT_gate(q, q + 1)
        for q in range(QUBIT_COUNT):
            circuit.add_RY_gate(q, parameters[parameter_index])
            parameter_index += 1

    if measure:
        bits = list(range(QUBIT_COUNT))
        circuit.measure(bits, bits)
    return circuit


state_preparation_circuit = build_h4_ry_ansatz(ZERO_PARAMETERS)
assert state_preparation_circuit.qubit_count == 8
assert len(state_preparation_circuit.gates) == 132  # X×4 + ansatz 128
state_preparation_circuit

## 2. 回路構造を確認する

`show_circuit()` で、QURI Partsが生成した回路を表示します。最初の4本のXゲートがHartree–Fock配置を作り、その後に72本の $R_y$ ゲートと56本のCNOTゲートが続きます。各繰返しが、`q0` から `q7` へつながる7本のCNOT ladderと8本の $R_y$ ゲートで構成されていることを確認します。

ここでは状態の値ではなく、ゲートの種類、順序、接続、繰返し回数を回路の設計と照合します。

### 実行イメージ
<img src="doc/image/qni.show_circuit.png" width="700">


In [3]:
qni.show_circuit(state_preparation_circuit)

QniViewer(url='http://127.0.0.1:48697/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B0%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2

## 3. 期待する状態を確認する

`show_circuit_and_state()` は、回路と、選択した位置まで実行した状態ベクトルを並べて表示します。回路上の縦棒を1つずつ数える必要はありません。状態を確認したい位置の縦棒を、画面上で直接選択します。

まず回路の最初の位置で、Hartree–Fock配置 `00001111` が100%になっていることを確認します。次に回路の最後、測定直前の位置を選び、全回転角を0にしたチェック用条件で同じ配置に戻ることを確認します。

途中の位置では、CNOT ladderによって複数の計算基底状態が現れます。この回路は各ゲート位置で電子数や $S_z$ を保つ設計ではないため、途中で別の電子数やスピンの状態が表示されても、それだけで実装ミスとは判断しません。ここで確認するのは、既知の初期状態、最終状態、そして回路の各位置での状態変化です。

VQEで最適化したパラメーターを使う場合は、最終状態が複数の電子配置の重ね合わせになります。その場合は、特定のビット列に戻ることではなく、状態ベクトルの分布を確認します。電子数などの条件に合わない成分は、QSCIの後段で行う事後選別の対象になります。

### 実行イメージ

<img src="doc/image/qni.show_circuit_and_state.png" width="700">


In [4]:
qni.show_circuit_and_state(state_preparation_circuit)

QniViewer(url='http://127.0.0.1:48697/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B0%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2

### 判定基準

| 選択位置 | 期待する表示 | 確認内容 |
|---|---|---|
| 回路の最初 | `00001111`：100% | 4電子・$S_z=0$ のHartree–Fock配置が作られている |
| 回路の最後（測定直前） | `00001111`：100% | 全回転角0のdepth 8回路が既知の最終状態に戻る |

この基準は、回転角をすべて0にしたチェック用条件にだけ適用します。VQEで最適化したパラメーターでは複数配置の重ね合わせになるため、最終状態が同じビット列である必要はありません。

## 4. 測定を含む回路を確認する

最後に、8量子ビットを計算基底で測定する回路を表示します。全回転角を0にしたチェック用条件では、測定結果は `00001111` になります。8量子ビットが同じ番号の古典bitへ対応付けられていることも確認します。

論文の10,000 shotsによるサンプリング、頻度上位 $R=1,4,16,27$ の配置選択、$N_e=4, S_z=0$ によるpost-selection、部分空間ハミルトニアンの古典対角化は `qsci_h4_paper_reproduction.ipynb` で扱います。このNotebookの状態表示や1回の測定結果は、論文のQSCIサンプリング結果そのものではありません。

### 実行イメージ
右にスクロールし、測定結果を確認
<img src="doc/image/qni.show_circuit(sampling_circuit).png" width="700">


In [6]:
sampling_circuit = build_h4_ry_ansatz(ZERO_PARAMETERS, measure=True)
qni.show_circuit(sampling_circuit)

QniViewer(url='http://127.0.0.1:48697/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B0%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2

## 5. このデモで確認できること

このデモでは、QSCI原論文Section IVのH₄実験で使われた8量子ビット・depth 8の $R_y$ ansatzをQURI Partsで構築し、VQEの最適化とサンプリングへ進む前に、その構造と状態を確認しました。全回転角を0にしたチェック用条件ではHartree–Fock配置に戻ることを、回路上の位置と状態ベクトルを対応付けて確認できます。

### ワークフローとの関係

例としてあげた論文のQSCIでは、
- (1) VQEなどで入力状態を準備
- (2) 計算基底で測定
- (3) 保存量でpost-selectionを行う
- (4) 頻度の高い配置を選ぶ
- (5) 選択した部分空間のハミルトニアンを古典対角化する

という流れです。このNotebookが扱うのは、そのうち(1)の回路を構築した直後、最適化と測定へ渡す前の確認です。QSCIそのものに新しい処理を加えるのではなく、入力回路を確認する工程をワークフローの境界に置いています。

論文との対応は、8量子ビット、4電子、Hartree–Fock初期状態、depth 8の $R_y$ ansatz、独立した回転パラメーター、隣接CNOT ladderです。一方、BFGSによる最適化、10,000-shotの分布、device noise、QSCIの配置選択とエネルギー計算は再現していません。そのため、これはFig. 8の結果再現ではなく、結果を得る前の入力回路を確認するデモです。

### このデモでまだ分からないこと

QniNotebookを使うことでレビュー時間や理解度が実際に改善するか、また通常の確認方法より有用かは、このデモだけでは判断できません。回路の誤りを見つける作業をQniNotebookあり／なしで比較し、発見率、原因箇所の特定時間、説明に必要なやり取りを利用者評価で調べる必要があります。

このデモは、8量子ビットまでの完全状態ベクトルによる回路構築確認です。QSCIによる配置選択やエネルギー計算は行いません。

## 6. QniNotebookの利用範囲

このデモでは、8量子ビットまでの回路を読み取り専用で表示します。状態表示には完全状態ベクトル simulationを使うため、大規模回路や実機での性能を扱うものではありません。

回路の編集、Pythonコードへの書き戻し、VQEの最適化、10,000-shotのサンプリング、QSCIの配置選択や古典対角化は、このNotebookの対象外です。QniNotebookが解釈できない回路を受け取った場合は、別の意味の回路として表示せず、理由を示して処理を停止します。

以上が、このデモで確認できる機能と、論文のQSCI計算に引き渡す前の位置づけです。